# RSNA Knee Abnormality Detection — Phase 4E submission rehearsal

Internet-off Kaggle notebook. **Do not enable internet. GPU (Tesla T4) is required.**
This notebook runs **visible/example-test inference only** with the five frozen
gold58 v2 fold checkpoints. It does **not** train, does **not** process
report-only studies, does **not** run the second reader, does **not** touch
hidden-test labels, and does **not** create a Kaggle competition submission.

Create a **new** kernel slug `mahdimzafari/rsna-knee-gold58-submission-rehearsal-v1`.
Do **not** overwrite frozen kernels (gold-pilot v7, constrained-canary v3,
gold-constrained-v14 v1, gold58-image-oof-v1 v1, gold58-image-oof-v2 v1,
notebooks 07/09/10/11).

`HARD_WEAK_LABEL_TRAINING_ALLOWED=false`.
`SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=false`.
`SECOND_READER_SCORE_BEARING_ALLOWED=false`.
`gold_second_reader_rerun_authorized=false`.


In [ ]:
# Bootstrap: stdlib only until the package is importable.
# Copies/extracts source under /kaggle/working. Never writes under /kaggle/input.
from __future__ import annotations

import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
SRC_DST = KAGGLE_WORKING / "rsna_knee_src"
OUTPUT_DIR = KAGGLE_WORKING / "rsna_gold58_submission_rehearsal"
MARKERS = ("knee/config.py", "knee/phase4e_rehearsal.py")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TORCH_HOME"] = str(KAGGLE_WORKING / "torch_cache")


def _looks_like_source(root: Path) -> bool:
    return all((root / marker).is_file() for marker in MARKERS)


def _safe_extract(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    dest_resolved = dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not name or name.endswith("/"):
                continue
            target = (dest / name).resolve()
            try:
                target.relative_to(dest_resolved)
            except ValueError as exc:
                raise RuntimeError(f"refusing zip path that escapes {dest}: {name}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, target.open("wb") as out:
                shutil.copyfileobj(src, out)


def _copy_allowlisted(src_root: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for dirname in ("knee", "scripts", "configs", "notebooks", "vendor", "third_party"):
        src = src_root / dirname
        if src.is_dir():
            shutil.copytree(src, dest / dirname, dirs_exist_ok=True)
    for filename in ("pyproject.toml", "requirements.txt", "README.md", "literature/ontology.yaml"):
        src = src_root / filename
        if src.is_file():
            dest_file = dest / filename
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest_file)


def _zip_has_source(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False
    return any(any(name.endswith(marker) for marker in MARKERS) for name in names)


SKIP_WALK_DIRS = {"train_series", "test_series", "__pycache__", ".git", "competitions"}


def _walk_input_for_source(root: Path, max_depth: int = 6):
    zips = []
    dirs = []
    try:
        root_res = root.resolve()
    except OSError:
        return zips, dirs
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        try:
            rel = current.resolve().relative_to(root_res)
            depth = 0 if str(rel) == "." else len(rel.parts)
        except (OSError, ValueError):
            dirnames[:] = []
            continue
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_WALK_DIRS and d != COMPETITION_SLUG)
        if depth > max_depth:
            dirnames[:] = []
            continue
        if _looks_like_source(current):
            dirs.append(current)
        for name in filenames:
            if name.lower().endswith(".zip"):
                zip_path = current / name
                if _zip_has_source(zip_path):
                    zips.append(zip_path)
    return zips, dirs


def locate_and_extract_source():
    if _looks_like_source(SRC_DST):
        return SRC_DST
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _looks_like_source(candidate):
            if KAGGLE_WORKING.exists() and candidate != SRC_DST:
                _copy_allowlisted(candidate, SRC_DST)
                if _looks_like_source(SRC_DST):
                    return SRC_DST
            return candidate
    if not KAGGLE_INPUT.is_dir():
        return None
    zips = []
    dirs = []
    for child in sorted(KAGGLE_INPUT.iterdir()):
        if child.name == COMPETITION_SLUG or child.name == "competitions":
            continue
        if child.is_file() and child.suffix.lower() == ".zip" and _zip_has_source(child):
            zips.append(child)
        elif child.is_dir():
            found_zips, found_dirs = _walk_input_for_source(child)
            zips.extend(found_zips)
            dirs.extend(found_dirs)
    if zips:
        _safe_extract(zips[0], SRC_DST)
        nested = [p for p in SRC_DST.rglob("knee") if (p.parent / "knee" / "phase4e_rehearsal.py").is_file()]
        if _looks_like_source(SRC_DST):
            return SRC_DST
        if nested:
            return nested[0].parent
    if dirs:
        _copy_allowlisted(dirs[0], SRC_DST)
        if _looks_like_source(SRC_DST):
            return SRC_DST
        return dirs[0]
    return None


SOURCE_ROOT = locate_and_extract_source()
print("source_root=" + str(SOURCE_ROOT))
if SOURCE_ROOT is None:
    raise SystemExit("SOURCE_NOT_FOUND: attach mahdimzafari/cursor-knee-source")
sys.path.insert(0, str(SOURCE_ROOT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output_dir=" + str(OUTPUT_DIR))


In [ ]:
import json
import os
import time
from pathlib import Path

from knee.phase4e_rehearsal import KERNEL_SLUG, run_phase4e_rehearsal
from knee.kaggle_runtime import KaggleLayout, preflight
from knee.phase4d_readiness import SECOND_READER_SCORE_BEARING_ALLOWED
from knee.report_labels import (
    HARD_WEAK_LABEL_TRAINING_ALLOWED,
    SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED,
)

print("kernel_slug=" + KERNEL_SLUG)
print("kernel_slug_literal=mahdimzafari/rsna-knee-gold58-submission-rehearsal-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-pilot")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-constrained-canary")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-second-reader-gold-constrained-v14")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v1")
print("do_not_overwrite_frozen_kernel=mahdimzafari/rsna-knee-gold58-image-oof-v2")
print("notebook_07_executed=false")
print("notebook_09_executed=false")
print("notebook_10_executed=false")
print("notebook_11_executed=false")
print("gold_second_reader_rerun_authorized=false")
print("HARD_WEAK_LABEL_TRAINING_ALLOWED=" + str(HARD_WEAK_LABEL_TRAINING_ALLOWED).lower())
print("SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED=" + str(SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED).lower())
print("SECOND_READER_SCORE_BEARING_ALLOWED=" + str(SECOND_READER_SCORE_BEARING_ALLOWED).lower())
print("DO_NOT_SUBMIT")

assert HARD_WEAK_LABEL_TRAINING_ALLOWED is False
assert SCALE_TO_REPORT_ONLY_SECOND_READER_ALLOWED is False
assert SECOND_READER_SCORE_BEARING_ALLOWED is False

layout = KaggleLayout(output_dirname="rsna_gold58_submission_rehearsal")
pre = preflight(layout=layout, output_dir=OUTPUT_DIR)
print("Preflight passed. Analysis will read CSVs and a bounded DICOM header sample only. Pixel arrays are limited to the documented Level C sample. Internet is not used. GPU is not required. Inputs are read-only.")
print("data_source=", pre.data_source if hasattr(pre, "data_source") else "")

try:
    import torch
    if torch.cuda.is_available():
        print("gpu_name=" + str(torch.cuda.get_device_name(0)))
    else:
        print("gpu_name=")
except Exception as exc:
    print("gpu_name=")
    print("gpu_check_error=" + type(exc).__name__)

t0 = time.time()
result = run_phase4e_rehearsal(
    output_dir=OUTPUT_DIR,
    execute_inference=True,
    allow_tiny_encoder_for_tests=False,
    verify_frozen_hashes=True,
    require_cuda=True,
    layout=layout,
    internet_enabled=False,
)
wall = time.time() - t0
public = {
    "status": result.get("status"),
    "ok": result.get("ok"),
    "kernel_slug": result.get("kernel_slug"),
    "n_visible_test_studies": result.get("n_visible_test_studies"),
    "n_report_only_studies_processed": result.get("n_report_only_studies_processed"),
    "competition_submission_written": result.get("competition_submission_written"),
    "consumes_submission_slot": result.get("consumes_submission_slot"),
    "hidden_test_available": result.get("hidden_test_available"),
    "training_run": result.get("training_run"),
    "series_selection_aggregates": result.get("series_selection_aggregates"),
    "rehearsal_submission": result.get("rehearsal_submission"),
    "runtime_guard": result.get("runtime_guard"),
    "deterministic_repeat": result.get("deterministic_repeat"),
    "wall_seconds": result.get("wall_seconds") or wall,
    "peak_gpu_memory_bytes": result.get("peak_gpu_memory_bytes"),
    "gpu_name": result.get("gpu_name"),
    "checkpoint_sha256": result.get("checkpoint_sha256"),
    "config_sha256": result.get("config_sha256"),
}
print("public=" + json.dumps(public, sort_keys=True))
print("PHASE4E_DONE status=" + str(result.get("status")))
print("DO_NOT_SUBMIT")
if Path("/kaggle/working/submission.csv").exists():
    raise SystemExit("REFUSING_WORKING_ROOT_SUBMISSION_CSV")
